# 02 - Forensic Features

Per-document linguistic indicators:
- Gunning-Fog readability index
- Type-Token Ratio (lexical diversity)
- Obfuscation Flag (binary: high Fog AND low TTR, cf. Lyon and Maxwell, 2011)
- Controversy Density (Loughran-McDonald Litigious terms per 1,000 words)
- Social Pillar Index (VADER sentiment on GRI 401-405 paragraphs)
- Circularity Density (Ellen MacArthur Foundation terms per 1,000 words)

All thresholds computed at runtime from the corpus distribution.

In [1]:
%run 00_config.ipynb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


18:46:40 [INFO] VERIS -- Project root: E:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20
18:46:40 [INFO] VERIS --   [OK] data/raw/reports: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\reports
18:46:40 [INFO] VERIS --   [OK] data/processed/text: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\processed\text
18:46:40 [INFO] VERIS --   [OK] outputs/csv: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\csv
18:46:40 [INFO] VERIS --   [OK] outputs/figures: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\figures
18:46:40 [INFO] VERIS --   [OK] climate_trace/DATA: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\climate_trace\DATA
18:46:40 [INFO] VERIS -- 

In [2]:
corpus = load_corpus_from_text(cfg.TEXT_DIR)

18:46:40 [INFO] VERIS -- Corpus loaded: 119 documents from 12 firms


In [3]:
# Gunning-Fog Index and Type-Token Ratio.
def count_syllables(word):
    """Approximate syllable count by counting vowel groups."""
    word = word.lower().strip()
    if not word:
        return 0
    vowels = "aeiouy"
    count = 0
    prev_was_vowel = False
    for ch in word:
        is_vowel = ch in vowels
        if is_vowel and not prev_was_vowel:
            count += 1
        prev_was_vowel = is_vowel
    if word.endswith("e") and count > 1:
        count -= 1
    return max(1, count)

def gunning_fog_index(text):
    """Gunning-Fog = 0.4 * (words/sentences + 100 * complex_words/words)."""
    sentences = re.split(r"[.!?]+", text)
    sentences = [s for s in sentences if s.strip()]
    words = re.findall(r"\b[a-zA-Z]+\b", text)
    if not sentences or not words:
        return 0.0
    complex_words = sum(1 for w in words if count_syllables(w) >= 3)
    avg_sentence_len = len(words) / len(sentences)
    pct_complex = 100 * complex_words / len(words)
    return round(0.4 * (avg_sentence_len + pct_complex), 4)

def type_token_ratio(text):
    """Unique words divided by total words."""
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    if not words:
        return 0.0
    return round(len(set(words)) / len(words), 4)

In [4]:
# Controversy Density using LM Litigious category.
# word-boundary regex for accurate term matching
# word boundary), causing `words` to be empty and the function to return 0 for every
# document. Replaced with single-backslash raw-string `\b` which compiles correctly.
def build_controversy_regex():
    """Compile a word-boundary regex for the LM Litigious vocabulary from nb00."""
    terms = sorted({t.lower() for t in CONTROVERSY_TERMS}, key=len, reverse=True)
    pattern = r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b"
    return re.compile(pattern, re.IGNORECASE), len(terms)

_controversy_re, _lm_n_terms = build_controversy_regex()
log.info(f"Controversy regex built over {_lm_n_terms} terms")

def controversy_density(text):
    """Return LM Litigious hits per 1,000 words."""
    words = re.findall(r"\b\w+\b", text)
    if not words:
        return 0.0
    hits = len(_controversy_re.findall(text))
    return round(1000 * hits / len(words), 4)

18:46:40 [INFO] VERIS -- Controversy regex built over 905 terms


In [5]:
# Social Pillar Index. VADER on sentences containing GRI 401-405 workforce keywords.
# Tier B: window_chars = 300 gives roughly 50-60 words of context around each keyword,
# matching a typical paragraph span. Empirically stable against ranges 150-500.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_vader = SentimentIntensityAnalyzer()

def social_pillar_index(text, window_chars=300):
    """Mean VADER compound score on text windows around GRI 401-405 keywords."""
    kw_pattern = re.compile(
        r"\b(" + "|".join(re.escape(k) for k in SOCIAL_KEYWORDS) + r")\b",
        re.IGNORECASE,
    )
    matches = list(kw_pattern.finditer(text))
    if not matches:
        return 0.0, 0
    scores = []
    for m in matches:
        lo = max(0, m.start() - window_chars)
        hi = min(len(text), m.end() + window_chars)
        snippet = text[lo:hi]
        sc = _vader.polarity_scores(snippet)
        scores.append(sc["compound"])
    return round(float(np.mean(scores)), 4), len(scores)

In [6]:
# Circularity Density. EMF Glossary terms per 1,000 words.
def build_circularity_regex():
    terms = sorted({t.lower() for t in CIRCULARITY_VOCAB}, key=len, reverse=True)
    pattern = r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b"
    return re.compile(pattern, re.IGNORECASE), len(terms)

_circ_re, _circ_n = build_circularity_regex()
log.info(f"Circularity regex built over {_circ_n} terms")

def circularity_density(text):
    words = re.findall(r"\b\w+\b", text)
    if not words:
        return 0.0
    hits = len(_circ_re.findall(text))
    return round(1000 * hits / len(words), 4)

18:46:40 [INFO] VERIS -- Circularity regex built over 23 terms


In [7]:
# Run per-document feature extraction.
def compute_forensic_features(corpus):
    """Return DataFrame with one row per (firm, year) document."""
    rows = []
    for (firm, year), text in tqdm(corpus.items(), desc="Forensic features"):
        soc_score, soc_n = social_pillar_index(text)
        rows.append({
            "firm_name": firm,
            "year": year,
            "gunning_fog_index": gunning_fog_index(text),
            "type_token_ratio":  type_token_ratio(text),
            "controversy_density": controversy_density(text),
            "social_index": soc_score,
            "n_social_paragraphs": soc_n,
            "circularity_density": circularity_density(text),
            "word_count": len(re.findall(r"\b\w+\b", text)),
        })
    return pd.DataFrame(rows)

forensic = compute_forensic_features(corpus)

# Obfuscation Flag: high Fog (p75) AND low TTR (p25). Thresholds from corpus at runtime.
fog_p75 = forensic["gunning_fog_index"].quantile(0.75)
ttr_p25 = forensic["type_token_ratio"].quantile(0.25)
log.info(f"Obfuscation thresholds computed from corpus: Fog p75 = {fog_p75:.3f}, TTR p25 = {ttr_p25:.3f}")

forensic["obfuscation_flag"] = (
    (forensic["gunning_fog_index"] >= fog_p75) &
    (forensic["type_token_ratio"]  <= ttr_p25)
).astype(int)

n_flagged = forensic["obfuscation_flag"].sum()
log.info(f"Obfuscation flag raised on {n_flagged} / {len(forensic)} documents")

forensic.to_csv(cfg.FORENSIC_CSV, index=False)
log.info(f"Forensic features saved: {cfg.FORENSIC_CSV.name} ({len(forensic)} rows)")
display(forensic.head(15))

Forensic features: 100%|██████████| 119/119 [00:51<00:00,  2.33it/s]
18:47:31 [INFO] VERIS -- Obfuscation thresholds computed from corpus: Fog p75 = 19.764, TTR p25 = 0.130
18:47:31 [INFO] VERIS -- Obfuscation flag raised on 9 / 119 documents
18:47:31 [INFO] VERIS -- Forensic features saved: forensic_features.csv (119 rows)


,firm_name,year,gunning_fog_index,type_token_ratio,controversy_density,social_index,n_social_paragraphs,circularity_density,word_count,obfuscation_flag
0,BP,2014,16.6653,0.1366,4.4470,0.5919,349,0.4118,24286,0
1,BP,2015,16.7015,0.1355,3.0537,0.5662,348,0.4539,24233,0
2,BP,2016,16.3946,0.1395,2.0243,0.5128,371,0.2892,24206,0
3,BP,2017,15.8763,0.1414,2.2000,0.3724,365,0.1222,24545,0
4,BP,2018,17.1948,0.1373,1.7964,0.5974,381,0.2089,23937,0
5,BP,2019,17.2420,0.1266,0.9131,0.7382,296,0.4150,24095,0
6,BP,2020,18.2982,0.1253,1.1982,0.6276,240,0.6790,25037,0
7,BP,2021,18.3889,0.1265,1.6287,0.7574,322,0.2036,24559,0
8,BP,2022,17.4171,0.1200,1.4635,0.6819,252,0.2769,25282,0
9,BP,2023,18.1969,0.1239,1.9827,0.6286,310,0.2065,24210,0
